# Phase 6 — Explainable AI for the Tri-Modal Model

**Thesis:** Tri-Modal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University

## Why explainability is non-negotiable in clinical AI

> A clinician will not act on a black-box depression flag. They need to see the
> *evidence*: which facial muscles, which modality, which signal drove the decision.
> A model that cannot justify itself cannot be deployed — regardless of its AUC.

This notebook applies **three complementary, mutually-validating** XAI methods to
the trained fusion model. Crucially, when two *independent* methods agree, the
explanation is trustworthy — not an artifact of one technique.

| Method | Type | Answers |
|--------|------|---------|
| **Cross-modal attention rollout** | model-intrinsic | Which modality does the model rely on? |
| **Gradient × input saliency** | gradient-based | Which facial Action Units drive the prediction? |
| **Leave-one-modality-out ablation** | occlusion-based | Does removing a modality actually hurt AUC? |

*Grounded in Shrikumar et al. (ICML 2017) and Abnar & Zuidema (ACL 2020).*

In [ ]:
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

## Prerequisite
A trained model at `results/fusion_trimodal_best.pth` (from Phase 4).

In [ ]:
# Runs all three explanations and saves figures + JSON
!python src/explainability/explain_fusion.py

## Visualize the explanations

In [ ]:
from IPython.display import Image, display
print('Facial Action Unit importance (which muscles signal depression):')
display(Image('results/figures/phase6_au_saliency.png'))
print('Per-modality importance — two independent estimates should agree:')
display(Image('results/figures/phase6_modality_importance.png'))

## Inspect the structured explanation report

In [ ]:
import json
with open('results/metrics/phase6_explainability.json') as f:
    rep = json.load(f)
print('Full-model dev AUC:', rep['full_model_auc'])
print('\nModality importance (attention):',
      rep['attention_rollout']['modality_importance'])
print('\nTop facial Action Units:')
for au in rep['top_action_units']:
    print(f"  {au['au']:6s} {au['clinical']:28s} {au['importance']}")
print('\nAblation AUC-drop per modality:', rep['ablation_auc_drop'])

## How to read this for the thesis defence

- **Convergent validity:** if attention rollout and ablation rank the same
  modality as most important, the model's reliance is *real*, not cosmetic.
- **Clinical face check:** the FACS units flagged with `*` (AU04 Brow-Lowerer,
  AU15 Lip-Corner-Depressor, AU17 Chin-Raiser) are the muscle movements
  established in affective-science literature as markers of sad/depressed affect.
  Seeing them rank highly is evidence the model learned *clinically plausible*
  features rather than spurious correlations.
- **Honesty note:** on the small CPU baseline, attributions are indicative, not
  definitive. The same code produces sharper, more stable attributions once the
  model is trained on GPU with the full DAIC-WOZ corpus.